<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
ChromaDB &amp; Indexing
</b></font> </br></p>

---

In [1]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"]    = "M12-ChromaDB-Indexing"
os.environ["LANGSMITH_ENDPOINT"]   = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment, get_ipinfo, setup_api_keys,
    mprint, install_packages, mermaid, load_prompt,
    show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

✓ OPENAI_API_KEY erfolgreich gesetzt
✓ LANGSMITH_API_KEY erfolgreich gesetzt

Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.2.12
langchain-chroma                         1.1.0
langchain-classic                        1.0.3
langchain-community                      0.4.1
langchain-core                           1.2.19
langchain-ollama                         1.0.1
langchain-openai                         1.1.11
langchain-text-splitters                 1.1.1
langgraph                                1.1.2
langgraph-checkpoint                     4.0.1
langgraph-prebuilt                       1.0.8
langgraph-sdk                            0.3.11

IP-Adresse: 34.145.41.191
Hostname: 191.41.145.34.bc.googleusercontent.com
Stadt: The Dalles
Region: Oregon
Land: US
Koordinaten: 45.5946,-121.1787
Provider: AS396982 Google LLC
Postleitzahl: 97058
Zeitzone: America/Los_Angeles


In [ ]:
#@title 📦 Pakete installieren{ display-mode: "form" }

install_packages([
                ('markitdown[all]', 'markitdown'),
                'langchain_huggingface',
                ('unstructured[all-docs]>=0.11.2', 'unstructured'),
                ])

# 1 | Übersicht
---

**ChromaDB** ist eine quelloffene Vektordatenbank, die speziell für KI-Anwendungen entwickelt wurde.  
Sie speichert **Texte als Vektoren** und ermöglicht blitzschnelle Ähnlichkeitssuche.

## Warum ChromaDB?

| Feature | ChromaDB | Alternativen |
|---------|---------|-------------|
| Installation | `pip install chromadb` | Faiss: C-Build nötig |
| Persistenz | Dateisystem (SQLite) | Pinecone: Cloud-only |
| Metadaten-Filter | ✅ Eingebaut | Weaviate: komplexer |
| LangChain-Integration | ✅ Nahtlos | Alle unterstützt |
| Lokaler Betrieb | ✅ Vollständig | Pinecone: nur Cloud |

## In diesem Modul

1. ChromaDB einrichten (in-memory & persistent)
2. Dokumente indexieren (laden, chunken, embedden)
3. Collections verwalten (erstellen, filtern, aktualisieren)
4. Häufige Fehler und Lösungen

In [2]:
#@title
#@markdown   <p><font size="4" color='green'>  flowchart</font> </br></p>

diagram = '''
flowchart LR
    subgraph Indexierung
        DOCS["📄 Dokumente"] --> LOADER["📥 Document Loader"]
        LOADER --> SPLITTER["✂️ Text Splitter"]
        SPLITTER --> EMBED["🔢 OpenAI Embeddings"]
        EMBED --> CHROMA[("🗄️ ChromaDB")]
    end
    subgraph Suche
        Q["❓ Query"] --> QE["🔢 Query Embed"]
        QE --> SIM["🔍 Similarity Search"]
        CHROMA --> SIM
        SIM --> RESULT["📋 Top-K Docs"]
    end
'''

mermaid(diagram, width=1100)

# 2 | ChromaDB Setup
---

ChromaDB kann auf zwei Arten betrieben werden:

| Modus | Beschreibung | Einsatz |
|-------|-------------|--------|
| **In-Memory** | Daten nur im RAM | Schnelle Demos, Tests |
| **Persistent** | Daten auf Dateisystem | Produktion, Notebooks |

**Empfehlung:** Immer persistent verwenden – dann gehen Daten beim Neustart nicht verloren.

In [4]:
import chromadb
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# Verzeichnis fuer persistenten Speicher
CHROMA_DIR = "/chroma_m12"

# Embedding-Modell
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# ChromaDB persistent initialisieren
vectorstore = Chroma(
    collection_name="agenten_docs",
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR
)

print("ChromaDB initialisiert")
print(f"  Speicherort: {CHROMA_DIR}")
print(f"  Collection:  agenten_docs")
print(f"  Dokumente:   {vectorstore._collection.count()}")

ChromaDB initialisiert
  Speicherort: /chroma_m12
  Collection:  agenten_docs
  Dokumente:   0


# 3 | Dokumente indexieren
---



Das Indexieren umfasst drei Schritte:

1. **Laden** – Dokumente einlesen (Text, PDF, Markdown, ...)
2. **Chunken** – In kleinere Abschnitte aufteilen
3. **Einbetten** – Vektoren erstellen und in ChromaDB speichern


<img src="https://raw.githubusercontent.com/ralf-42/Agenten/main/07_image/rag_process_01.png" width="650" alt="Indexierungs-Pipeline: Dokumente → Text Splitter → Embeddings → ChromaDB">

*Indexierungs-Pipeline: Dokumente → Text Splitter → Embeddings → ChromaDB*


<p><font color='black' size="5">
Dokument-Loader in LangChain
</font></p>


| Loader | Dateityp | Import |
|--------|---------|--------|
| `TextLoader` | .txt | `langchain_community.document_loaders` |
| `PyPDFLoader` | .pdf | `langchain_community.document_loaders` |
| `UnstructuredMarkdownLoader` | .md | `langchain_community.document_loaders` |
| `DirectoryLoader` | Verzeichnis | `langchain_community.document_loaders` |

<img src="https://raw.githubusercontent.com/ralf-42/Agenten/main/07_image/rag_process_02.png" width="650" alt="Phase 2 – Retrieval: Query-Embedding → Similarity Search → Top-K Dokumente">

*Phase 2 – Retrieval: Query-Embedding → Similarity Search → Top-K Dokumente*

In [5]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Beispiel-Dokumente (Agenten-Wissensbasis) inline erstellen
dokumente_raw = [
    Document(
        page_content=(
            "LangChain ist ein Framework fuer die Entwicklung von KI-Agenten. "
            "Es bietet Werkzeuge fuer Tool-Use, Chains und Agenten-Orchestrierung. "
            "Die aktuelle Version 1.0 verwendet LCEL-Chains und init_chat_model()."
        ),
        metadata={"source": "langchain_info.txt", "thema": "frameworks"}
    ),
    Document(
        page_content=(
            "LangGraph ist ein Framework fuer Multi-Agent-Systeme. "
            "Es basiert auf gerichteten Graphen (StateGraphs) mit Knoten und Kanten. "
            "Checkpointing ermoeglicht persistente Agenten-Sessions."

        ),
        metadata={"source": "langgraph_info.txt", "thema": "frameworks"}
    ),
    Document(
        page_content=(
            "RAG steht fuer Retrieval-Augmented Generation. "
            "Es kombiniert Sprachmodelle mit Vektordatenbanken. "
            "ChromaDB ist eine beliebte Vektordatenbank fuer lokale RAG-Systeme."

        ),
        metadata={"source": "rag_info.txt", "thema": "rag"}
    ),
    Document(
        page_content=(
            "OpenAI bietet verschiedene Embedding-Modelle an. "
            "text-embedding-3-small hat 1536 Dimensionen und ist kostenguenstig. "
            "text-embedding-3-large hat 3072 Dimensionen und ist genauer."

        ),
        metadata={"source": "embeddings_info.txt", "thema": "embeddings"}
    ),
]

# Text Splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=30,
)
chunks = splitter.split_documents(dokumente_raw)

print(f"Geladene Dokumente: {len(dokumente_raw)}")
print(f"Chunks nach Splitting: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"  Chunk {i+1}: {len(chunk.page_content)} Zeichen | Quelle: {chunk.metadata.get('source', '?')} | Thema: {chunk.metadata.get('thema', '?')}")

Geladene Dokumente: 4
Chunks nach Splitting: 4
  Chunk 1: 204 Zeichen | Quelle: langchain_info.txt | Thema: frameworks
  Chunk 2: 181 Zeichen | Quelle: langgraph_info.txt | Thema: frameworks
  Chunk 3: 165 Zeichen | Quelle: rag_info.txt | Thema: rag
  Chunk 4: 177 Zeichen | Quelle: embeddings_info.txt | Thema: embeddings


In [6]:
# Chunks in ChromaDB indexieren
vectorstore.add_documents(chunks)

print(f"Indexiert! Dokumente in ChromaDB: {vectorstore._collection.count()}")

Indexiert! Dokumente in ChromaDB: 4


# 4 | Collections verwalten
---



Eine **Collection** in ChromaDB ist eine benannte Gruppe von Dokumenten – vergleichbar mit einer Tabelle in SQL.



<p><font color='black' size="5">
Wichtige Operationen
</font></p>


| Operation | Methode | Beschreibung |
|-----------|---------|-------------|
| Dokument hinzufügen | `add_documents()` | Neue Chunks indexieren |
| Ähnlichkeitssuche | `similarity_search()` | Top-K ähnliche Dokumente |
| Mit Score | `similarity_search_with_score()` | Suche + Ähnlichkeitswert |
| Metadaten-Filter | `filter={...}` | Nur bestimmte Dokumente |
| Alle Dokumente | `get()` | Rohe Collection-Daten |

In [7]:
# Einfache Aehnlichkeitssuche mit Query-Rewrite aus Prompt-Template
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

llm_rewrite = init_chat_model("openai:gpt-4o-mini", temperature=0.0)
rewrite_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m12_query_rewrite_prompt.md", mode="T")

frage_raw = "Welche Frameworks gibt es fuer KI-Agenten?"
frage = (rewrite_prompt | llm_rewrite | StrOutputParser()).invoke({"user_question": frage_raw}).strip()

ergebnisse = vectorstore.similarity_search(frage, k=2)

print(f"Originalfrage: {frage_raw}")
print(f"Retrieval-Query: {frage}")
print()
print("Top-2 Ergebnisse:")
for i, doc in enumerate(ergebnisse):
    print(f"{i+1}. {doc.page_content[:120]}...")


HTTPError: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/ralf-42/Agenten/main/05_prompt/m12_query_rewrite_prompt.md

In [ ]:
# Suche mit Ähnlichkeitsscore
ergebnisse_mit_score = vectorstore.similarity_search_with_score(
    "Was ist ChromaDB?",
    k=3
)

mprint("## Suchergebnis mit Scores\n")
rows = []
for doc, score in ergebnisse_mit_score:
    rows.append(f"| {doc.metadata.get('thema', '?')} | {score:.4f} | {doc.page_content[:80]}... |")

mprint("| Thema | Score | Inhalt (Vorschau) |\n|-------|-------|-----------------|\n" + "\n".join(rows))

In [ ]:
# Metadaten-Filter: Nur RAG-Dokumente suchen
rag_ergebnisse = vectorstore.similarity_search(
    "Wie funktioniert Retrieval?",
    k=2,
    filter={"thema": "rag"}  # Nur Dokumente mit thema=rag
)

print("Gefilterte Suche (nur thema=rag):")
for doc in rag_ergebnisse:
    print(f"  - {doc.metadata['source']}: {doc.page_content[:100]}...")

# 5 | ChromaDB Troubleshooting
---

## Häufige Fehler und Lösungen

| Problem | Symptom | Lösung |
|---------|---------|--------|
| **Doppelte Indexierung** | Dokumente mehrfach vorhanden | IDs prüfen, Collection neu aufbauen |
| **Falsches persist_directory** | Daten weg nach Neustart | Absoluten Pfad verwenden |
| **Falsche Collection** | Leere Suchergebnisse | `collection_name` prüfen |
| **Embedding-Modell-Mismatch** | Dimension Error | Immer gleiches Modell für Index & Query |
| **Zu wenige Chunks** | k > Anzahl Dokumente | k ≤ Anzahl indexierter Chunks setzen |

## Best Practices

```python
# IMMER: Gleiche Embeddings für Index und Query
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")  # Einmal definieren

# IMMER: Absoluten Pfad für persist_directory
import os
CHROMA_DIR = os.path.abspath("../02_daten/05_sonstiges/chroma_db")

# TIPP: Vorhandene Collection prüfen vor erneutem Indexieren
count = vectorstore._collection.count()
if count == 0:
    vectorstore.add_documents(chunks)
else:
    print(f"Collection hat bereits {count} Eintraege.")
```

In [ ]:
# Diagnose: Collection-Status prüfen
count = vectorstore._collection.count()
print(f"Dokumente in Collection: {count}")

# Alle Metadaten anzeigen
raw = vectorstore._collection.get(include=["metadatas"])
themen = set(m.get("thema", "?") for m in raw["metadatas"])
print(f"Vorhandene Themen: {themen}")

# Tipp: Collection leeren und neu aufbauen
# vectorstore._collection.delete(where={"thema": "alt"})  # Selektiv löschen
print("\nCollection-Diagnose abgeschlossen.")

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M12-ChromaDB-Indexing", limit=3, show_steps=True)

# A | Aufgabe
---


<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs dürfen und sollen Sie generative KI auch als Unterstützung beim Lernen und Entwickeln nutzen. Wenn Sie bei einer Aufgabe festhängen, können Sie zum Beispiel Gemini in Google Colab verwenden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.



<p><font color='black' size="5">
Wissensdatenbank für KI-Agenten
</font></p>

Bauen Sie eine persistente Wissensdatenbank für das Agenten-Kurs-Material auf.

**Teilaufgaben:**
1. Laden Sie mindestens 3 Textdokumente aus `../02_daten/01_text/` (biografien_*.txt, *.md)
2. Chunken Sie die Dokumente mit `RecursiveCharacterTextSplitter` (chunk_size=400, overlap=40)
3. Indexieren Sie die Chunks in einer ChromaDB-Collection (persistent)
4. Führen Sie 3 verschiedene Suchanfragen durch und zeigen Sie die Ergebnisse als Markdown-Tabelle

**Bonus:** Nutzen Sie Metadaten-Filter, um die Suche auf eine bestimmte Dateiquelle einzuschränken.